
# Notebook 4: Feature Engineering

## Project: ThreatTrace — CICIDS2017 Intrusion Detection

### Objective
Prepare the cleaned CICIDS2017 dataset for subsequent machine learning experiments by examining its features and identifying opportunities for feature engineering and selection.

### Input Dataset
`data/processed/cicids2017_cleaned.csv`

### Planned Activities
1. Load and validate the cleaned dataset.
2. Inspect feature names and data types.
3. Separate input features from the target label.
4. Identify constant and duplicate features.
5. Examine feature distributions and relationships.
6. Decide which feature transformations or removals are justified.
7. Save the prepared dataset and document the findings.

### Scope
This notebook focuses on feature analysis and preparation. Class balancing, model training, and model evaluation will be handled in later notebooks.


In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pathlib import Path
import hashlib

import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:

# ============================================================
# 2. LOAD CLEANED DATASET
# ============================================================

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data/processed/cicids2017_cleaned.csv").exists():
    PROJECT_ROOT = Path("D:/ThreatTrace")

DATA_PATH = PROJECT_ROOT / "data/processed/cicids2017_cleaned.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Check the project directory and Notebook 3 output."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

display(df.head())


Dataset loaded successfully.
Dataset shape: (2517883, 79)
Number of rows: 2,517,883
Number of columns: 79
Memory usage: 1.59 GB


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,33,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,1.100917e+05,18348.62385,109.0,0.0,109,109,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,9174.311927,9174.311927,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,2.307692e+05,38461.53846,52.0,0.0,52,52,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,19230.769230,19230.769230,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,3.529412e+05,58823.52941,34.0,0.0,34,34,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,29411.764710,29411.764710,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,31,329,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:

# ============================================================
# 3. DATASET VALIDATION
# ============================================================

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.value_counts().rename_axis("Data Type").to_frame("Count"))

print("\nMissing values:", int(df.isna().sum().sum()))

numeric_df = df.select_dtypes(include=[np.number])

print(
    "Infinite values:",
    int(np.isinf(numeric_df.to_numpy()).sum())
)

print("Duplicate rows:", int(df.duplicated().sum()))

if "Label" not in df.columns:
    raise ValueError(
        "Target column 'Label' was not found. "
        "Inspect the column names before continuing."
    )

print("\nTarget label distribution:")
display(
    df["Label"]
    .value_counts(dropna=False)
    .rename_axis("Label")
    .to_frame("Number of Records")
)


Dataset shape: (2517883, 79)

Column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Fl

,Count
Data Type,
int64,54
float64,24
str,1



Missing values: 0
Infinite values: 0
Duplicate rows: 0

Target label distribution:


,Number of Records
Label,
BENIGN,2092335
DoS Hulk,172688
DDoS,127995
PortScan,90694
DoS GoldenEye,10281
FTP-Patator,5927
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3217



## 4. Feature Quality Analysis

In this section, we examine the input features to identify:
- Constant features with no variation.
- Duplicate feature columns.
- Numerical and non-numerical columns.
- Features that may require further preprocessing.

The target column `Label` is excluded from feature-quality checks.


In [4]:

# ============================================================
# 4. FEATURE QUALITY ANALYSIS
# ============================================================

# Separate target from input features
target_col = "Label"

feature_cols = [col for col in df.columns if col != target_col]

print("Total columns:", df.shape[1])
print("Target column:", target_col)
print("Number of input features:", len(feature_cols))

# Separate numerical and non-numerical features
numeric_features = df[feature_cols].select_dtypes(
    include=[np.number]
).columns.tolist()

non_numeric_features = df[feature_cols].select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\nNumerical features:", len(numeric_features))
print("Non-numerical features:", len(non_numeric_features))

print("\nNon-numerical feature names:")
print(non_numeric_features)

# Identify constant features
constant_features = [
    col for col in feature_cols
    if df[col].nunique(dropna=False) <= 1
]

print("\nConstant features:", len(constant_features))
print(constant_features)

Total columns: 79
Target column: Label
Number of input features: 78

Numerical features: 78
Non-numerical features: 0

Non-numerical feature names:
[]

Constant features: 8
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']



## 6. Removal of Constant Features

The feature-quality analysis identified eight constant features:

1. Bwd PSH Flags
2. Bwd URG Flags
3. Fwd Avg Bytes/Bulk
4. Fwd Avg Packets/Bulk
5. Fwd Avg Bulk Rate
6. Bwd Avg Bytes/Bulk
7. Bwd Avg Packets/Bulk
8. Bwd Avg Bulk Rate

Constant features contain only one unique value across the dataset. They provide no discriminative information and can be removed to reduce the input dimensionality.

The target column `Label` is retained because it contains the classification categories required for intrusion detection.


In [5]:

# ============================================================
# 6. REMOVE CONSTANT FEATURES
# ============================================================

# Store the original feature count
original_feature_count = len(feature_cols)

# Remove constant features from the input data
df_features = df.drop(columns=constant_features).copy()

# Keep the target column unchanged
assert "Label" in df_features.columns
assert df_features["Label"].equals(df["Label"])

# Calculate the updated feature count
remaining_features = [
    col for col in df_features.columns if col != "Label"
]

print("Original input features:", original_feature_count)
print("Constant features removed:", len(constant_features))
print("Remaining input features:", len(remaining_features))
print("Total columns after removal:", df_features.shape[1])

print("\nRemoved features:")
print(constant_features)

print("\nUpdated dataset shape:", df_features.shape)


Original input features: 78
Constant features removed: 8
Remaining input features: 70
Total columns after removal: 71

Removed features:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Updated dataset shape: (2517883, 71)


## 7. Duplicate Feature Detection

After removing constant features, the remaining input features
are checked for exact duplicate columns.

Duplicate feature columns contain identical values across all
records, even if their column names are different.

Keeping duplicate columns can unnecessarily increase the
feature dimensionality without adding new information.

The target column `Label` is excluded from this analysis.

A duplicate column will be identified by comparing its hash
with previously processed columns and then confirming equality
using an exact column comparison.

If duplicate features are found, the first occurrence will be
retained and subsequent identical columns will be removed.

This step is separate from correlation analysis. Highly
correlated features are not necessarily exact duplicates.

In [6]:
# ============================================================
# 7. DUPLICATE FEATURE DETECTION
# ============================================================

# Input features after constant-feature removal
features_to_check = [
    col for col in df_features.columns
    if col != "Label"
]

# Store previously encountered feature columns by hash
seen_feature_hashes = {}

# Store detected duplicate pairs
duplicate_feature_pairs = []

for col in features_to_check:

    # Calculate a hash for the entire feature column
    column_hashes = pd.util.hash_pandas_object(
        df_features[col],
        index=False
    ).to_numpy(dtype="uint64")

    # Generate a compact digest
    digest = hashlib.blake2b(
        column_hashes.tobytes(),
        digest_size=16
    ).hexdigest()

    # Check columns having the same digest
    candidates = seen_feature_hashes.setdefault(
        digest, []
    )

    matched_column = None

    for previous_col in candidates:

        # Exact comparison to guard against hash collisions
        if df_features[col].equals(df_features[previous_col]):
            matched_column = previous_col
            break

    if matched_column is not None:

        duplicate_feature_pairs.append(
            (matched_column, col)
        )

    else:
        candidates.append(col)


# Extract duplicate columns to remove
duplicate_features = [
    duplicate_col
    for original_col, duplicate_col
    in duplicate_feature_pairs
]

print("Duplicate feature pairs detected:",
      len(duplicate_feature_pairs))

print("\nDuplicate feature pairs:")

for original_col, duplicate_col in duplicate_feature_pairs:
    print(f"{original_col} <-> {duplicate_col}")


# Remove duplicate feature columns, retaining first occurrence
df_features = df_features.drop(
    columns=duplicate_features
).copy()


# Recalculate remaining input features
remaining_features = [
    col for col in df_features.columns
    if col != "Label"
]

print("\nDuplicate features removed:",
      len(duplicate_features))

print("Remaining input features:",
      len(remaining_features))

print("Updated dataset shape:",
      df_features.shape)

Duplicate feature pairs detected: 5

Duplicate feature pairs:
Fwd PSH Flags <-> SYN Flag Count
Fwd URG Flags <-> CWE Flag Count
Fwd Header Length <-> Fwd Header Length.1
Total Fwd Packets <-> Subflow Fwd Packets
Total Backward Packets <-> Subflow Bwd Packets

Duplicate features removed: 5
Remaining input features: 65
Updated dataset shape: (2517883, 66)


## 8. Verification of Remaining Features

After removing constant and exact duplicate features,
the remaining dataset is validated.

The verification checks:

- No constant input features remain.
- No exact duplicate input features remain.
- No missing values exist.
- No infinite numerical values exist.
- The target column `Label` is preserved.
- All target classes remain available.

In [7]:
# ============================================================
# 8. VERIFY REMAINING FEATURES
# ============================================================

# Check remaining constant features
remaining_constant_features = [
    col for col in remaining_features
    if df_features[col].nunique(dropna=False) <= 1
]

print("Remaining constant features:",
      len(remaining_constant_features))

print(remaining_constant_features)


# Verify duplicate detection results from Section 7
assert len(duplicate_features) == len(duplicate_feature_pairs)

print("\nDuplicate feature columns removed:",
      len(duplicate_features))

print("Remaining exact duplicate features: 0")


# Check missing values
missing_count = int(df_features.isna().sum().sum())
print("\nMissing values:", missing_count)


# Check infinite numerical values
numeric_remaining = df_features[
    remaining_features
].select_dtypes(include=[np.number])

infinite_count = int(
    np.isinf(numeric_remaining.to_numpy()).sum()
)

print("Infinite values in input features:",
      infinite_count)


# Verify target
assert "Label" in df_features.columns

print("\nTarget class count:",
      df_features["Label"].nunique())

print("Target distribution:")

display(
    df_features["Label"]
    .value_counts()
    .rename_axis("Label")
    .to_frame("Records")
)


# Final assertions
assert len(remaining_constant_features) == 0
assert missing_count == 0
assert infinite_count == 0

print("\nFeature verification completed successfully.")


Remaining constant features: 0
[]

Duplicate feature columns removed: 5
Remaining exact duplicate features: 0

Missing values: 0
Infinite values in input features: 0

Target class count: 15
Target distribution:


,Records
Label,
BENIGN,2092335
DoS Hulk,172688
DDoS,127995
PortScan,90694
DoS GoldenEye,10281
FTP-Patator,5927
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3217



Feature verification completed successfully.



## 9. Feature Correlation Analysis

This section examines relationships between numerical input features using Pearson correlation.

Highly correlated features may contain overlapping information. We will identify pairs with strong absolute correlation and inspect them before deciding whether any further feature reduction is justified.

The target column `Label` is excluded from the correlation calculation. No features will be removed based on correlation alone in this step.


In [8]:

# ============================================================
# 8. FEATURE CORRELATION ANALYSIS
# ============================================================

# Select numerical input features only
X_numeric = df_features.drop(columns=["Label"]).select_dtypes(
    include=[np.number]
)

print("Number of numerical input features:", X_numeric.shape[1])

# Calculate the correlation matrix
correlation_matrix = X_numeric.corr()

print("Correlation matrix shape:", correlation_matrix.shape)

# Extract unique feature pairs with absolute correlation >= 0.95
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

high_correlation_pairs = (
    upper_triangle.stack()
    .rename("correlation")
    .reset_index()
)

high_correlation_pairs.columns = [
    "Feature 1", "Feature 2", "Correlation"
]

high_correlation_pairs = high_correlation_pairs[
    high_correlation_pairs["Correlation"].abs() >= 0.95
].sort_values(
    by="Correlation",
    key=lambda values: values.abs(),
    ascending=False
)

print(
    "\nNumber of feature pairs with absolute correlation >= 0.95:",
    len(high_correlation_pairs)
)

display(high_correlation_pairs.head(30))


Number of numerical input features: 65
Correlation matrix shape: (65, 65)

Number of feature pairs with absolute correlation >= 0.95: 40


,Feature 1,Feature 2,Correlation
569,Fwd Packet Length Mean,Avg Fwd Segment Size,1.000000
830,Bwd Packet Length Mean,Avg Bwd Segment Size,1.000000
377,Total Length of Bwd Packets,Subflow Bwd Bytes,1.000000
311,Total Length of Fwd Packets,Subflow Fwd Bytes,0.999999
85,Flow Duration,Fwd IAT Total,0.998542
1193,Flow IAT Max,Fwd IAT Max,0.998077
2518,Packet Length Mean,Average Packet Size,0.997835
2776,RST Flag Count,ECE Flag Count,0.997801
228,Total Backward Packets,Bwd Header Length,0.993898
133,Total Fwd Packets,Total Backward Packets,0.991594


In [9]:
# Display correlation results

correlation_threshold = 0.95

print(
    "\nNumber of feature pairs with absolute "
    f"correlation >= {correlation_threshold}:",
    len(high_correlation_pairs)
)

print("\nTop highly correlated feature pairs:")

display(
    high_correlation_pairs.head(30)
)

print("\nCorrelation analysis completed.")


Number of feature pairs with absolute correlation >= 0.95: 40

Top highly correlated feature pairs:


,Feature 1,Feature 2,Correlation
569,Fwd Packet Length Mean,Avg Fwd Segment Size,1.000000
830,Bwd Packet Length Mean,Avg Bwd Segment Size,1.000000
377,Total Length of Bwd Packets,Subflow Bwd Bytes,1.000000
311,Total Length of Fwd Packets,Subflow Fwd Bytes,0.999999
85,Flow Duration,Fwd IAT Total,0.998542
1193,Flow IAT Max,Fwd IAT Max,0.998077
2518,Packet Length Mean,Average Packet Size,0.997835
2776,RST Flag Count,ECE Flag Count,0.997801
228,Total Backward Packets,Bwd Header Length,0.993898
133,Total Fwd Packets,Total Backward Packets,0.991594



Correlation analysis completed.


## 10. Interpretation of Feature Correlation

The correlation analysis examines the numerical input features using Pearson correlation.

The latest previously recorded execution identified 65 feature pairs with an absolute correlation of at least 0.95. This count must be reconfirmed after executing the updated notebook because exact duplicate feature removal occurs before correlation analysis.

### Key Findings

- Several features are perfectly or almost perfectly correlated.
- Some features represent equivalent or closely related network traffic measurements.
- Strong correlation may indicate overlapping information.
- Correlation does not establish causation.
- Highly correlated features are not automatically unnecessary for classification.

### Decision

Eight constant features were removed during feature-quality analysis.

Exact duplicate feature columns are checked separately and subsequent duplicate columns are removed while retaining the first occurrence.

The remaining features are preserved at this stage because correlation alone is not sufficient justification for removing them. Further feature selection will be evaluated using training data and machine learning experiments.

Class imbalance remains significant and will be addressed during model development.


## 11. Derived Feature Engineering Experiment

In this section, two additional network-flow features are
created to represent directional traffic characteristics.

### 1. Forward Packet Share

This feature represents the proportion of packets transmitted
in the forward direction relative to the total number of
forward and backward packets.

### 2. Forward Byte Share

This feature represents the proportion of bytes transmitted
in the forward direction relative to the total transferred
bytes.

### Handling Zero Denominators

Some network flows may have zero total packet counts or zero
total transferred bytes.

To avoid division-by-zero errors, these cases are assigned
a value of zero. This convention is documented because zero
represents an undefined ratio in these specific cases.

### Experimental Decision

The original feature-engineered dataset will be retained as
the baseline.

The dataset containing the two additional features will be
saved separately.

Both datasets will later be compared using the same
training and validation split to determine whether the
derived features improve classification performance.

In [10]:
# ============================================================
# 11. DERIVED FEATURE ENGINEERING EXPERIMENT
# ============================================================

# Create a separate experimental dataset
df_features_experiment = df_features.copy()


# ------------------------------------------------------------
# 1. Forward Packet Share
# ------------------------------------------------------------

total_packets = (
    df_features_experiment["Total Fwd Packets"]
    + df_features_experiment["Total Backward Packets"]
)

df_features_experiment["Fwd_Packet_Share"] = np.divide(
    df_features_experiment["Total Fwd Packets"],
    total_packets,
    out=np.zeros(len(df_features_experiment), dtype="float64"),
    where=total_packets.to_numpy() != 0
)


# ------------------------------------------------------------
# 2. Forward Byte Share
# ------------------------------------------------------------

total_bytes = (
    df_features_experiment["Total Length of Fwd Packets"]
    + df_features_experiment["Total Length of Bwd Packets"]
)

df_features_experiment["Fwd_Byte_Share"] = np.divide(
    df_features_experiment["Total Length of Fwd Packets"],
    total_bytes,
    out=np.zeros(len(df_features_experiment), dtype="float64"),
    where=total_bytes.to_numpy() != 0
)


# ------------------------------------------------------------
# Display experimental features
# ------------------------------------------------------------

new_features = [
    "Fwd_Packet_Share",
    "Fwd_Byte_Share"
]

print("New features created:", len(new_features))
print(new_features)

display(
    df_features_experiment[
        new_features
    ].describe()
)


# ------------------------------------------------------------
# Validate derived features
# ------------------------------------------------------------

for col in new_features:

    assert df_features_experiment[col].notna().all()
    assert np.isfinite(
        df_features_experiment[col].to_numpy()
    ).all()

    assert df_features_experiment[col].between(
        0, 1
    ).all()


# Verify original target remains unchanged
assert df_features_experiment["Label"].equals(
    df_features["Label"]
)

print("\nDerived feature validation completed successfully.")

print("Baseline shape:", df_features.shape)
print("Experimental shape:", df_features_experiment.shape)

New features created: 2
['Fwd_Packet_Share', 'Fwd_Byte_Share']


,Fwd_Packet_Share,Fwd_Byte_Share
count,2.517883e+06,2.517883e+06
mean,5.782592e-01,3.323362e-01
std,1.806594e-01,3.154426e-01
min,6.369427e-03,0.000000e+00
25%,5.000000e-01,5.539036e-02
50%,5.000000e-01,2.603550e-01
75%,6.000000e-01,4.347826e-01
max,1.000000e+00,1.000000e+00



Derived feature validation completed successfully.
Baseline shape: (2517883, 66)
Experimental shape: (2517883, 68)


In [11]:
# Define output path

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_PATH = (
    OUTPUT_DIR / "cicids2017_feature_engineered.csv"
)

print("Output path:", OUTPUT_PATH)

Output path: D:\ThreatTrace\data\processed\cicids2017_feature_engineered.csv


## 10. Interpretation of Feature Correlation

The correlation analysis examined 70 numerical input features
using Pearson correlation.

A total of 65 feature pairs were identified with an absolute
correlation value greater than or equal to 0.95.

### Key Findings

- Several features are perfectly or almost perfectly correlated.
- Some features represent equivalent or closely related network
  traffic measurements.
- Strong correlation may indicate overlapping information.
- Correlation does not establish causation.
- Highly correlated features are not automatically unnecessary
  for classification.

### Decision

Eight constant features were removed during feature-quality
analysis.

Exact duplicate feature columns were also checked separately.

The remaining features are preserved at this stage because
correlation alone is not sufficient justification for removing
features.

Further feature selection will be evaluated using training
data and machine learning experiments.

Class imbalance remains significant, particularly for rare
attack classes. It will be addressed separately during model
development.

## 11. Derived Feature Engineering Experiment

In this section, two additional network-flow features are
created to represent directional traffic characteristics.

### 1. Forward Packet Share

This feature represents the proportion of packets transmitted
in the forward direction relative to the total number of
forward and backward packets.

### 2. Forward Byte Share

This feature represents the proportion of bytes transmitted
in the forward direction relative to the total transferred
bytes.

### Handling Zero Denominators

Some network flows may have zero total packet counts or zero
total transferred bytes.

To avoid division-by-zero errors, these cases are assigned
a value of zero. This convention is documented because zero
represents an undefined ratio in these specific cases.

### Experimental Decision

The original feature-engineered dataset will be retained as
the baseline.

The dataset containing the two additional features will be
saved separately.

Both datasets will later be compared using the same
training and validation split to determine whether the
derived features improve classification performance.

In [12]:
# ============================================================
# 11. DERIVED FEATURE ENGINEERING EXPERIMENT
# ============================================================

# Create a separate experimental dataset
df_features_experiment = df_features.copy()


# ------------------------------------------------------------
# 1. Forward Packet Share
# ------------------------------------------------------------

total_packets = (
    df_features_experiment["Total Fwd Packets"]
    + df_features_experiment["Total Backward Packets"]
)

df_features_experiment["Fwd_Packet_Share"] = np.divide(
    df_features_experiment["Total Fwd Packets"],
    total_packets,
    out=np.zeros(len(df_features_experiment), dtype="float64"),
    where=total_packets.to_numpy() != 0
)


# ------------------------------------------------------------
# 2. Forward Byte Share
# ------------------------------------------------------------

total_bytes = (
    df_features_experiment["Total Length of Fwd Packets"]
    + df_features_experiment["Total Length of Bwd Packets"]
)

df_features_experiment["Fwd_Byte_Share"] = np.divide(
    df_features_experiment["Total Length of Fwd Packets"],
    total_bytes,
    out=np.zeros(len(df_features_experiment), dtype="float64"),
    where=total_bytes.to_numpy() != 0
)


# ------------------------------------------------------------
# Display experimental features
# ------------------------------------------------------------

new_features = [
    "Fwd_Packet_Share",
    "Fwd_Byte_Share"
]

print("New features created:", len(new_features))
print(new_features)

display(
    df_features_experiment[
        new_features
    ].describe()
)


# ------------------------------------------------------------
# Validate derived features
# ------------------------------------------------------------

for col in new_features:

    assert df_features_experiment[col].notna().all()
    assert np.isfinite(
        df_features_experiment[col].to_numpy()
    ).all()

    assert df_features_experiment[col].between(
        0, 1
    ).all()


# Verify original target remains unchanged
assert df_features_experiment["Label"].equals(
    df_features["Label"]
)

print("\nDerived feature validation completed successfully.")

print("Baseline shape:", df_features.shape)
print("Experimental shape:", df_features_experiment.shape)

New features created: 2
['Fwd_Packet_Share', 'Fwd_Byte_Share']


,Fwd_Packet_Share,Fwd_Byte_Share
count,2.517883e+06,2.517883e+06
mean,5.782592e-01,3.323362e-01
std,1.806594e-01,3.154426e-01
min,6.369427e-03,0.000000e+00
25%,5.000000e-01,5.539036e-02
50%,5.000000e-01,2.603550e-01
75%,6.000000e-01,4.347826e-01
max,1.000000e+00,1.000000e+00



Derived feature validation completed successfully.
Baseline shape: (2517883, 66)
Experimental shape: (2517883, 68)


## 12. Save Feature-Engineered Datasets

Two datasets are saved:

1. Baseline dataset:
   Contains the remaining original features after constant
   and duplicate feature removal.

2. Experimental dataset:
   Contains the baseline features and two additional
   derived network-flow features.

Both datasets retain the original target column `Label`.

The baseline dataset will be used for comparison against
the experimental dataset during model development.

In [13]:
# ============================================================
# 12. SAVE FEATURE-ENGINEERED DATASETS
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Save baseline dataset
# ------------------------------------------------------------

BASELINE_PATH = (
    OUTPUT_DIR / "cicids2017_feature_engineered.csv"
)

df_features.to_csv(
    BASELINE_PATH,
    index=False
)

print("Baseline dataset saved successfully.")
print("Path:", BASELINE_PATH)
print("Shape:", df_features.shape)
print(
    "File size:",
    f"{BASELINE_PATH.stat().st_size / (1024**3):.2f} GB"
)


# ------------------------------------------------------------
# Save experimental dataset
# ------------------------------------------------------------

EXPERIMENT_PATH = (
    OUTPUT_DIR / "cicids2017_feature_engineered_experiment.csv"
)

df_features_experiment.to_csv(
    EXPERIMENT_PATH,
    index=False
)

print("\nExperimental dataset saved successfully.")
print("Path:", EXPERIMENT_PATH)
print("Shape:", df_features_experiment.shape)
print(
    "File size:",
    f"{EXPERIMENT_PATH.stat().st_size / (1024**3):.2f} GB"
)

Baseline dataset saved successfully.
Path: D:\ThreatTrace\data\processed\cicids2017_feature_engineered.csv
Shape: (2517883, 66)
File size: 0.76 GB

Experimental dataset saved successfully.
Path: D:\ThreatTrace\data\processed\cicids2017_feature_engineered_experiment.csv
Shape: (2517883, 68)
File size: 0.81 GB


In [14]:
# ============================================================
# 13. VERIFY SAVED DATASETS
# ============================================================

# Check baseline dataset
assert BASELINE_PATH.exists()
assert BASELINE_PATH.stat().st_size > 0

# Check experimental dataset
assert EXPERIMENT_PATH.exists()
assert EXPERIMENT_PATH.stat().st_size > 0


# Verify baseline shape
assert len(df_features) == len(df)
assert "Label" in df_features.columns

# Verify experimental shape
assert len(df_features_experiment) == len(df_features)
assert "Label" in df_features_experiment.columns

# Verify target labels are unchanged
assert df_features["Label"].equals(df["Label"])

assert df_features_experiment["Label"].equals(
    df_features["Label"]
)


# Verify new features
for col in new_features:
    assert col in df_features_experiment.columns

    assert df_features_experiment[col].between(
        0, 1
    ).all()


print("BASELINE DATASET")
print("----------------")
print("Rows:", len(df_features))
print("Input features:", len(remaining_features))
print("Total columns:", len(df_features.columns))
print("Target classes:", df_features["Label"].nunique())


print("\nEXPERIMENTAL DATASET")
print("--------------------")
print("Rows:", len(df_features_experiment))
print(
    "Input features:",
    len(df_features_experiment.columns) - 1
)
print("Total columns:", len(df_features_experiment.columns))
print("Additional features:", new_features)


print("\nAll dataset verification checks passed.")

BASELINE DATASET
----------------
Rows: 2517883
Input features: 65
Total columns: 66
Target classes: 15

EXPERIMENTAL DATASET
--------------------
Rows: 2517883
Input features: 67
Total columns: 68
Additional features: ['Fwd_Packet_Share', 'Fwd_Byte_Share']

All dataset verification checks passed.


In [15]:
# ============================================================
# FINAL NOTEBOOK STATISTICS
# ============================================================

print("FINAL NOTEBOOK STATISTICS")
print("=========================")

print("Dataset records:", len(df_features))
print("Original input features:", len(feature_cols))
print("Constant features removed:", len(constant_features))
print("Duplicate features removed:", len(duplicate_features))
print("Remaining baseline features:", len(remaining_features))
print("Correlation pairs (|r| >= 0.95):", len(high_correlation_pairs))
print("Target classes:", df_features["Label"].nunique())
print("Baseline columns:", df_features.shape[1])
print("Experimental columns:", df_features_experiment.shape[1])

print("\nBaseline dataset:", BASELINE_PATH)
print("Experimental dataset:", EXPERIMENT_PATH)


FINAL NOTEBOOK STATISTICS
Dataset records: 2517883
Original input features: 78
Constant features removed: 8
Duplicate features removed: 5
Remaining baseline features: 65
Correlation pairs (|r| >= 0.95): 40
Target classes: 15
Baseline columns: 66
Experimental columns: 68

Baseline dataset: D:\ThreatTrace\data\processed\cicids2017_feature_engineered.csv
Experimental dataset: D:\ThreatTrace\data\processed\cicids2017_feature_engineered_experiment.csv


## 14. Final Conclusion

In this notebook, feature-quality analysis and an initial derived-feature experiment were performed on the cleaned CICIDS2017 dataset.

### Work Completed

1. Loaded and validated the cleaned CICIDS2017 dataset.
2. Examined numerical input features and target labels.
3. Identified and removed constant features.
4. Detected and removed exact duplicate input features, if present.
5. Verified the remaining features for missing values, infinite values, and constant columns.
6. Performed Pearson correlation analysis using an absolute correlation threshold of 0.95.
7. Preserved the target column `Label` and its class labels.
8. Created two experimental features:
   - `Fwd_Packet_Share`
   - `Fwd_Byte_Share`
9. Saved the baseline and experimental datasets separately.

### Final Dataset Information

The exact feature counts and correlation-pair count are printed by the final statistics cell after execution. These values are intentionally not hardcoded here because duplicate-feature detection may change the final number of baseline features.

#### Baseline Dataset

Output:
`data/processed/cicids2017_feature_engineered.csv`

Contains the original retained input features and the target column `Label`.

#### Experimental Dataset

Output:
`data/processed/cicids2017_feature_engineered_experiment.csv`

Contains the baseline features, two additional derived features, and the target column `Label`.

### Important Findings

Highly correlated feature pairs were identified, but features were not removed solely because of correlation. The two derived features represent forward-direction packet share and byte share. Their predictive usefulness has not yet been established.

### Limitations

- Class imbalance remains in the dataset.
- The usefulness of derived features must be tested through model comparison.
- Correlation does not establish predictive importance.
- Class balancing, model training, and model evaluation are outside the scope of this notebook.

### Next Steps

The next stage will compare the baseline and experimental feature sets using identical training and validation partitions. Evaluation will include macro F1-score, per-class recall, and false-positive rate.
